# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](Imagen.png)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [9]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10.0,   # entrega
            (2, 2): +2.0,    # carga
            (3, 5): -10.0,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9
        self.p_normal = 0.90             # prob. de la dirección elegida, piso normal
        self.p_slippery = 0.60           # prob. de la dirección elegida, piso resbaloso

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        # Terminal absorbente
        if self.is_terminal(state):
            return [(state, 1.0)]

        p_main = self.p_slippery if state in self.slippery_states else self.p_normal
        p_side = (1.0 - p_main) / 2

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        transitions = []
        for a, p in [(action, p_main), (perp1, p_side), (perp2, p_side)]:
            nxt = (state[0] + a[0], state[1] + a[1])
            if not self.is_valid_state(nxt):
                nxt = state
            transitions.append((nxt, p))
        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [10]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [11]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[s2] for s2, p in grid.get_transition_probs(state, action))


def _best_action(grid, state, V):
    # round() evita que ruido numérico rompa empates entre acciones
    return max(grid.actions,
               key=lambda a: round(expected_next_value(grid, state, a, V), 8))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for k in range(max_iter):
        V_new = V.copy()
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
                V_new[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V, k + 1


def extract_policy(grid, V):
    return {s: _best_action(grid, s, V)
            for s in grid.states() if not grid.is_terminal(s)}


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [12]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for _ in range(max_iter):
        V_new = V.copy()
        delta = 0.0
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(
                    grid, s, policy[s], V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return {s: _best_action(grid, s, V)
            for s in grid.states() if not grid.is_terminal(s)}


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold * 1e-3)  # evaluación más precisa
        new_policy = policy_improvement(grid, V)
        changes = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changes)
        if changes == 0:
            break
        policy = new_policy
    return policy, V, history


## Parte 4 — Visualización y comparación


In [13]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [14]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [15]:
def run(label, **kw):
    g = WarehouseMDP()
    for k, v in kw.items():
        setattr(g, k, v)
    V, _ = value_iteration(g)
    print(f"### {label}  |  V(start) = {V[g.start]:.3f}")
    print_policy(g, extract_policy(g, V))
    print()

run("Base")
run("Sin resbalón (p_slippery=0.9)", p_slippery=0.9)
run("A: living_reward=-0.1", living_reward=-0.1)
run("B: p_slippery=0.4", p_slippery=0.4)
run("C: gamma=0.99", gamma=0.99)

for lr in [-0.9, -0.85, -0.8, -0.75, -0.7]:   # bonus
    run(f"Bonus living_reward={lr}", living_reward=lr)

### Base  |  V(start) = -2.575
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

### Sin resbalón (p_slippery=0.9)  |  V(start) = -2.407
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

### A: living_reward=-0.1  |  V(start) = 1.649
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

### B: p_slippery=0.4  |  V(start) = -2.706
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

### C: gamma=0.99  |  V(start) = -1.456
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 

## Respuestas Parte 5

1. Desde START el robot prefiere la estación de carga +2: →, →, ↓ hasta (1,2) y ↓ a (2,2). V(START) = -2.575.
2. Una recompensa menor puede ser óptima porque la entrega +10 está lejos: el camino cruza el peligro -3 en (1,4), cada paso cuesta -1 y el futuro se descuenta con γ=0.9. La carga +2 es segura y cercana, y su valor neto es mayor.
3. El piso resbaloso cambia la decisión en (1,2): es ↓ (hacia la carga) con piso resbaloso y pasa a → (hacia la entrega) si el piso fuera normal. También cambia (3,1), que es normal pero está al lado de (2,1) resbaloso: con resbalón va → y sin él va ↑, porque evita pasar por la celda resbalosa.
4. El costo por paso -1 penaliza los caminos largos, así que favorece la recompensa cercana (+2) frente a la lejana (+10). Es la presión que hace "no tan malo" el +2.
5. Porque la dinámica depende del estado actual: en piso normal las probabilidades son 0.9/0.05/0.05 y en resbaloso 0.6/0.2/0.2. T(s,a,s') debe revisar si s está en `slippery_states`.

### Experimentos
- A (living_reward=-0.1): el robot cambia a buscar la entrega +10. En (1,2) va → y atraviesa la fila 1 hasta (1,5). Como cada paso cuesta poco, vale la pena el camino largo. V(START) = +1.649.
- B (p_slippery=0.4): la política no cambia (V(START) = -2.706). Ya con 0.6 el piso era lo bastante malo, y con 0.4 el robot sigue prefiriendo lo mismo, solo con peor valor.
- C (gamma=0.99): la política pasa a valorar la entrega +10. En (1,2) cambia ↓ por →. Con γ mayor, la recompensa lejana pesa más. V(START) = -1.456.
- **Bonus:** el cambio entre ir a carga e ir a entrega ocurre entre living_reward = -0.80 y -0.75 (aprox. -0.78). Con costos más negativos prefiere la carga, y con menos negativos va por la entrega.